In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch import nn
import numpy as np
from torch.utils.data import DataLoader

In [3]:
from src.architechures.cnn import Simple_CNN_1D
from src.trainers import SIM_CLR_Trainer
from src.metrics import get_scores, metrics
from src.datasets import DronesDataset, WiFiDataset

In [16]:
class Mlp(nn.Module):
    def __init__(self, in_features, out_features, apply_softmax = False):
        super(Mlp, self).__init__()
        self.ln1 = nn.Linear(in_features = in_features, out_features = in_features)
        self.relu = nn.ReLU()
        self.ln2 = nn.Linear(in_features = in_features, out_features = out_features)
        self.apply_softmax = apply_softmax
        self.softmax = nn.Softmax()
    def forward(self, x):
        out = self.ln2(self.relu(self.ln1(x)))
        if self.apply_softmax:
            out = self.softmax(out)
        return out

class Rotation(nn.Module):
    
    def __init__(self, in_channels, signal_length, noise_std=0):
        super(Rotation, self).__init__()
        self.in_channels = in_channels
        self.signal_length = signal_length

    def forward(self, x):
        # doesn't change the fingerprint
        angle = 2 * np.pi * np.random.choice([1., 0.75, 0.5, 0.25])                                     
        x = torch.stack(
            (x[...,0,:] * np.cos(angle) - x[...,1,:] * np.sin(angle),
            x[...,0,:] * np.sin(angle) +  x[...,1,:] * np.cos(angle)),
            dim = len(x.shape) - 2
            )

        return x

class Amplifier(nn.Module):
    
    def __init__(self, in_channels, signal_length, noise_std=0):
        super(Amplifier, self).__init__()
        self.in_channels = in_channels
        self.signal_length = signal_length
        self.noise_std = noise_std

    def forward(self, x):
        scale = torch.randn(1, device = x.device) * self.noise_std + 1

        x *= scale

        return x

class Bias(nn.Module):
    
    def __init__(self, in_channels, signal_length, noise_std=0):
        super(Bias, self).__init__()
        self.in_channels = in_channels
        self.signal_length = signal_length
        self.noise_std = noise_std

    def forward(self, x):
        bias = torch.randn(1, device = x.device) * self.noise_std

        x += bias

        return x

class Noise(nn.Module):
    
    def __init__(self, in_channels, signal_length, noise_std=0):
        super(Noise, self).__init__()
        self.in_channels = in_channels
        self.signal_length = signal_length
        self.noise_std = noise_std

    def forward(self, x):
        noise = torch.randn(x.shape, device = x.device) * self.noise_std

        x += noise

        return x
    
class Augmentation_Masked(nn.Module):
    def __init__(self, in_channels, singal_length, aug_module, noise_std = 0, num_bits = 10, prob = 0.5):
        
        super(Augmentation_Masked, self).__init__()

        assert singal_length % num_bits == 0

        self.size = in_channels * singal_length
        self.noise_std = noise_std
        self.num_bits = num_bits
        self.in_channels = in_channels
        self.singal_length = singal_length
        self.bit_size = singal_length // num_bits
        self.prob = prob


        self.layers = nn.ModuleList(
            [
                aug_module(in_channels, singal_length // num_bits, noise_std)

                for i in range(num_bits)
            ]
        )

    def forward(self, x):
        new_x = torch.zeros(x.shape, device = x.device)
        
        for i in range(self.num_bits):
            
            if np.random.choice([0,1], p = [1-self.prob, self.prob]):
                
                new_x[...,i * self.bit_size: (i+1) * self.bit_size] =\
                self.layers[i](
                    x[...,i * self.bit_size: (i+1) * self.bit_size]
                )
                
            else:
                
                new_x[...,i * self.bit_size: (i+1) * self.bit_size] =\
                    x[...,i * self.bit_size: (i+1) * self.bit_size]
        
        return new_x

In [17]:
def scale(array, mean = 1.5, std = 7239):
    return (array - mean) / std



def normalize_energy(tensor):
    if not torch.is_tensor(tensor):
        tensor = torch.tensor(tensor)
    
    energy = torch.sqrt((tensor**2).sum(0)).mean()
    
    return tensor / energy
    
dataset_train = DronesDataset('data_drones_500.h5', uavs=[1,2,3,4], bursts=[1],
                              transform=lambda x: normalize_energy(scale(x)))

dataset_test = DronesDataset('data_drones_500.h5', uavs=[1,2,3,4,5], bursts=[1],
                             transform=lambda x: normalize_energy(scale(x)) )


In [18]:
train_loader = DataLoader(dataset_train, batch_size = 128, num_workers = 3, shuffle=True)

test_loader = DataLoader(dataset_test, batch_size = 128, num_workers = 3, shuffle=False)

In [24]:
augs = nn.ModuleList(
    [Augmentation_Masked(2, 500, Rotation, num_bits = 50),
     Augmentation_Masked(2, 500, Amplifier, num_bits = 50, noise_std=0.3),
     Augmentation_Masked(2, 500, Noise, num_bits = 50, noise_std=0.3),
     Augmentation_Masked(2, 500, Bias, num_bits = 50, noise_std=0.3)
    ]
     
)
augs.to('cuda')
augs.train()

trainer = SIM_CLR_Trainer(augs)


In [25]:
model = Simple_CNN_1D(in_channels=2, input_signal_length=500, features_size = 100, maxplool_strides=3, kernel_size=10)
mlp_instance = Mlp(100, 5, False)
mpl_cluster = Mlp(100, 20, True)

output_size: 512


In [26]:
test_targets = []
for batch, target in test_loader:
    test_targets.append(target)
test_targets = torch.cat(test_targets)

In [28]:
optimizer = torch.optim.Adam([
            {'params': model.parameters(), 'lr': 1e-3}, 
            {'params': mlp_instance.parameters(), 'lr': 1e-3}, 
            {'params': mpl_cluster.parameters(), 'lr': 1e-3},
            {'params': augs.parameters(), 'lr': 0}
        ])

for epoch in range(20):
    
    loss = trainer.train_epoch(
        model, 
        mlp_instance, 
        mpl_cluster,
        train_loader, 
        test_loader,
        optimizer,
        'cuda'
    )
    
    if epoch % 5 == 0:
        train_f, test_f =\
        trainer.get_features(model, train_loader, test_loader, 'cuda',  mpl_cluster)
        print(loss)
        
        scores = get_scores(train_f, test_f, features_type='clusters_probas', clusters_num=1, criterion = 'left-sided')
        
        evals = metrics(1 - np.array(scores), test_targets > 4, treshold=0.95)
        evals['loss'] = loss
        
        print(epoch, evals)

7.485417572170275
0 {'roc_auc': 0.5547825395373762, 'f1': 0.10809135062519377, 'loss': 7.485417572170275}


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


7.483002077363103
5 {'roc_auc': 0.5500734873647113, 'f1': 0.0986602970194205, 'loss': 7.483002077363103}


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


7.476886215964569
10 {'roc_auc': 0.5200231879294555, 'f1': 0.08573818485989126, 'loss': 7.476886215964569}


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


7.471244095158574
15 {'roc_auc': 0.5196154498451591, 'f1': 0.09032123487692949, 'loss': 7.471244095158574}


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
